# ConsumerInsight AI — análise dos resultados

Notebook de apoio para explorar o dataset e avaliar execuções feitas por `main.py`. Ele não envia reclamações à API. O conjunto fornecido é sintético e os resultados não devem ser generalizados para reclamações reais.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "consumerinsight_reclamacoes_sinteticas.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src.data_processing import load_dataset
from src.evaluation import evaluate_results
from src.visualization import generate_charts

DATASET_PATH = PROJECT_ROOT / "consumerinsight_reclamacoes_sinteticas.csv"
RESULTS_PATH = PROJECT_ROOT / "outputs" / "reclamacoes_classificadas.csv"

## Dataset e rótulos de referência

Os campos de referência são úteis para avaliação local. Eles não fazem parte do prompt enviado à OpenRouter pelo classificador.

In [2]:
dataset = load_dataset(DATASET_PATH)
print(f"Registros: {len(dataset)}")
print(f"Categorias: {dataset['categoria_referencia'].nunique()}")
display(dataset[["categoria_referencia", "sentimento_referencia"]].value_counts().rename("quantidade").reset_index())

Registros: 720
Categorias: 9


,categoria_referencia,sentimento_referencia,quantidade
0,Atendimento ao Cliente,Negativo,64
1,Cadastro e Acesso,Negativo,64
2,Qualidade do Produto,Negativo,64
3,Cancelamento e Reembolso,Negativo,64
4,Entrega e Logistica,Negativo,64
5,Cobranca e Pagamentos,Negativo,64
6,Outros,Negativo,64
7,Servicos e Contratos,Negativo,64
8,Garantia e Assistencia Tecnica,Negativo,64
9,Atendimento ao Cliente,Neutro,16


## Resultados da IA

Execute `python main.py --limit 50` ou `python main.py` antes desta seção. A avaliação ignora registros sem previsão válida.

In [4]:
if not RESULTS_PATH.exists():
    print("Ainda não há resultados. Configure o .env e execute main.py.")
else:
    results = pd.read_csv(RESULTS_PATH, dtype=str, keep_default_na=False)
    print(results["status"].value_counts(dropna=False))
    evaluation = evaluate_results(results)
    display(evaluation["metrics"].style.format({
        "acuracia": "{:.2f}",
        "precisao_macro": "{:.2f}",
        "recall_macro": "{:.2f}",
        "f1_macro": "{:.2f}",
    }))
    for target, report in evaluation["reports"].items():
        print(f"Desempenho por classe — {target}")
        display(report.style.format({
            "precision": "{:.2f}",
            "recall": "{:.2f}",
            "f1-score": "{:.2f}",
            "support": "{:.0f}",
        }))
    chart_paths = generate_charts(
        results, evaluation, PROJECT_ROOT / "outputs" / "charts"
    )
    print(f"Gráficos gerados: {len(chart_paths)}")

status
success    30
Name: count, dtype: int64


,alvo,registros_avaliados,acuracia,precisao_macro,recall_macro,f1_macro
0,categoria,30,0.93,0.74,0.78,0.76
1,sentimento,30,0.80,0.27,0.33,0.30


Desempenho por classe — categoria


,classe,precision,recall,f1-score,support
0,Cobranca e Pagamentos,1.00,1.00,1.00,5
1,Entrega e Logistica,1.00,1.00,1.00,5
2,Qualidade do Produto,1.00,1.00,1.00,3
3,Atendimento ao Cliente,0.67,1.00,0.80,4
4,Cancelamento e Reembolso,1.00,1.00,1.00,5
5,Garantia e Assistencia Tecnica,1.00,1.00,1.00,5
6,Servicos e Contratos,0.00,0.00,0.00,0
7,Cadastro e Acesso,1.00,1.00,1.00,1
8,Outros,0.00,0.00,0.00,2
9,accuracy,0.93,0.93,0.93,1


Desempenho por classe — sentimento


,classe,precision,recall,f1-score,support
0,Negativo,0.80,1.00,0.89,24
1,Neutro,0.00,0.00,0.00,6
2,Positivo,0.00,0.00,0.00,0
3,accuracy,0.80,0.80,0.80,1
4,macro avg,0.27,0.33,0.30,30
5,weighted avg,0.64,0.80,0.71,30


Gráficos gerados: 8
